# 4b — Feature Engineering Ablation Study
## Forecasting with calendar features only (no history engineering)

This notebook runs the same models as **notebook 4**, but *without* the 10 history
features (lags, rolling statistics, change rates). Only the 9 calendar features
derived from the date are used.

| Setup | Features |
|---|---|
| **No FE** (this notebook) | 9 calendar features only |
| **With FE** (notebook 4) | 9 calendar + 10 history features |

**Purpose:** Quantify the contribution of feature engineering by comparing MAPE values.
Results are saved to `forecasting_model_results_no_features_avg_kwh/`


## 1. Imports & Configuration


In [ ]:
import warnings, json, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

try:
    import holidays as _holidays_lib
    _HAS_HOLIDAYS = True
except ImportError:
    _HAS_HOLIDAYS = False

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH   = Path(r"C:\Users\GONCA\Desktop\Iscte\MCD\Theses")
DATA_FILE   = BASE_PATH / "results" / "data" / "municipality_daily_consumption.csv"
if not DATA_FILE.exists():
    DATA_FILE = BASE_PATH / "municipality_daily_consumption.csv"
OUT_DIR     = BASE_PATH / "forecasting_model_results_no_features_avg_kwh"
OUT_DIR.mkdir(exist_ok=True)

# ── Config ────────────────────────────────────────────────────────────────────
TARGET_COL   = "avg_kwh"
DATE_COL     = "date"
GROUP_COL    = "municipality"
MUNICIPALITIES = ["Vitoria-Gasteiz", "Donostia/San Sebastian", "Pamplona/Iruna"]

TRAIN_FRAC      = 0.70
VAL_FRAC        = 0.15
SEQ_LEN         = 30
EPOCHS          = 80
BATCH_SIZE      = 64
PATIENCE        = 15
SEASONAL_PERIOD = 365
SARIMA_PERIOD   = 7
SEED            = 42

np.random.seed(SEED)

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Data:   {DATA_FILE}")
print(f"Output: {OUT_DIR}")


## 2. Calendar Feature Engineering

Only features derivable from the date are added — no consumption history.
The 9 calendar features used across all models:

| Feature | Description |
|---|---|
| `is_weekend` | 1 if Saturday or Sunday |
| `is_holiday_es` | 1 if Spanish national/regional holiday |
| `is_bridge_day` | 1 if Monday after / Friday before a holiday |
| `sin_dow`, `cos_dow` | Cyclic day-of-week encoding |
| `sin_month`, `cos_month` | Cyclic month encoding |
| `sin_week`, `cos_week` | Cyclic week-of-year encoding |


In [ ]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df["day_of_week"]  = df[DATE_COL].dt.dayofweek
    df["month"]        = df[DATE_COL].dt.month
    df["week_of_year"] = df[DATE_COL].dt.isocalendar().week.astype(int)
    df["is_weekend"]   = (df["day_of_week"] >= 5).astype(int)
    df["sin_dow"]   = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["cos_dow"]   = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["sin_month"] = np.sin(2 * np.pi * df["month"] / 12)
    df["cos_month"] = np.cos(2 * np.pi * df["month"] / 12)
    df["sin_week"]  = np.sin(2 * np.pi * df["week_of_year"] / 52)
    df["cos_week"]  = np.cos(2 * np.pi * df["week_of_year"] / 52)

    if _HAS_HOLIDAYS:
        years = sorted(df[DATE_COL].dt.year.unique())
        hols  = set(pd.to_datetime(
            list(_holidays_lib.country_holidays("ES", years=years).keys())))
        df["is_holiday_es"] = df[DATE_COL].isin(hols).astype(int)
        nd  = df[DATE_COL] + pd.Timedelta(days=1)
        pd_ = df[DATE_COL] - pd.Timedelta(days=1)
        df["is_bridge_day"] = (
            ((df["day_of_week"] == 0) & nd.isin(hols)) |
            ((df["day_of_week"] == 4) & pd_.isin(hols))
        ).astype(int)
    else:
        df["is_holiday_es"] = 0
        df["is_bridge_day"] = 0
    return df

CAL_FEATS = [
    "is_weekend", "is_holiday_es", "is_bridge_day",
    "sin_dow", "cos_dow", "sin_month", "cos_month", "sin_week", "cos_week",
]

print(f"Calendar features ({len(CAL_FEATS)}):", CAL_FEATS)


## 3. Metrics & Data Splitting


In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom != 0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100)

def metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    return mae, rmse, mape(y_true, y_pred), smape(y_true, y_pred)

def split_series(df):
    n    = len(df)
    n_tr = int(n * TRAIN_FRAC)
    n_va = int(n * VAL_FRAC)
    return df.iloc[:n_tr], df.iloc[n_tr:n_tr+n_va], df.iloc[n_tr+n_va:]

def _cal_array(df_sub, df_feats):
    cal = [f for f in CAL_FEATS if f in df_feats.columns]
    return df_feats.loc[df_sub.index, cal].values.astype(np.float32)


## 4. Baseline Models
Naive, Rolling Mean, and Seasonal Naive do not use any features.


In [ ]:
def run_naive(train_val, test):
    last = float(train_val[TARGET_COL].iloc[-1])
    return pd.Series(last, index=test.index)

def run_rolling_mean(train_val, test, window=7):
    last = float(train_val[TARGET_COL].iloc[-window:].mean())
    return pd.Series(last, index=test.index)

def run_seasonal_naive(train_val, test, period=SEASONAL_PERIOD):
    combined = pd.concat([train_val[[TARGET_COL]], test[[TARGET_COL]]])
    vals = combined[TARGET_COL].values
    n_tv = len(train_val)
    preds = [float(vals[n_tv + i - period]) if (n_tv + i - period) >= 0 else float(vals[0])
             for i in range(len(test))]
    return pd.Series(preds, index=test.index)


## 5. SARIMA
SARIMA uses only its own autoregressive history — no exogenous calendar features.


In [ ]:
def run_sarima(train, val, test):
    try:
        from statsmodels.tsa.statespace.sarimax import SARIMAX
        tv    = pd.concat([train, val])
        y     = tv[TARGET_COL].values
        use_log = float(y.min()) < 0.5
        y_fit = np.log1p(y.clip(min=0)) if use_log else y
        best_aic, best_order, best_sorder = np.inf, (1, 1, 1), (1, 1, 1, SARIMA_PERIOD)
        for p in [1, 2]:
            for q in [1, 2]:
                try:
                    m = SARIMAX(y_fit, order=(p, 1, q),
                                seasonal_order=(1, 1, 1, SARIMA_PERIOD),
                                enforce_stationarity=False,
                                enforce_invertibility=False).fit(disp=False, maxiter=150)
                    if m.aic < best_aic:
                        best_aic, best_order = m.aic, (p, 1, q)
                except Exception:
                    pass
        model = SARIMAX(y_fit, order=best_order,
                        seasonal_order=best_sorder,
                        enforce_stationarity=False,
                        enforce_invertibility=False).fit(disp=False, maxiter=150)
        raw       = model.forecast(len(test))
        pred_vals = np.expm1(np.array(raw)) if use_log else np.array(raw)
        return pd.Series(np.clip(pred_vals, 0, None), index=test.index)
    except Exception as e:
        print(f"  SARIMA failed: {e}")
        return pd.Series(np.nan, index=test.index)


## 6. Ridge & Random Forest (calendar features only)
Without history features, these models can only exploit
day-of-week / month / holiday calendar patterns.


In [ ]:
def run_ridge(train, val, test, df_feats):
    tv  = pd.concat([train, val])
    cal = [f for f in CAL_FEATS if f in df_feats.columns]
    sc  = StandardScaler()
    X_tr = sc.fit_transform(df_feats.loc[tv.index,    cal].values)
    X_te = sc.transform(    df_feats.loc[test.index,  cal].values)
    model = Ridge(alpha=1.0)
    model.fit(X_tr, tv[TARGET_COL].values)
    return pd.Series(model.predict(X_te).clip(0), index=test.index)

def run_rf(train, val, test, df_feats):
    tv  = pd.concat([train, val])
    cal = [f for f in CAL_FEATS if f in df_feats.columns]
    X_tr = df_feats.loc[tv.index,    cal].values
    X_te = df_feats.loc[test.index,  cal].values
    model = RandomForestRegressor(n_estimators=300, max_depth=12,
                                  min_samples_leaf=3,
                                  random_state=SEED, n_jobs=-1)
    model.fit(X_tr, tv[TARGET_COL].values)
    return pd.Series(model.predict(X_te).clip(0), index=test.index)


## 7. LSTM v2 (no history features)
Dual-stream architecture:
- **Sequential stream:** 30-day lookback of raw `avg_kwh` (1 feature, no engineering)
- **Static stream:** 9 calendar features only


In [ ]:
def run_lstm_v2(train, val, test, df_feats):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

    y_sc  = StandardScaler()
    y_tr  = y_sc.fit_transform(train[[TARGET_COL]]).ravel()
    y_va  = y_sc.transform(val[[TARGET_COL]]).ravel()
    y_te  = y_sc.transform(test[[TARGET_COL]]).ravel()

    cal_tr = _cal_array(train, df_feats)
    cal_va = _cal_array(val,   df_feats)
    cal_te = _cal_array(test,  df_feats)
    static_size = cal_tr.shape[1]

    class DualDS(Dataset):
        def __init__(self, y, cal, seq_len):
            self.y, self.cal, self.seq_len = y, cal, seq_len
        def __len__(self):
            return max(0, len(self.y) - self.seq_len)
        def __getitem__(self, i):
            return (torch.tensor(self.y[i:i+self.seq_len].reshape(-1, 1), dtype=torch.float32),
                    torch.tensor(self.cal[i+self.seq_len], dtype=torch.float32),
                    torch.tensor([self.y[i+self.seq_len]], dtype=torch.float32))

    class LSTMModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm   = nn.LSTM(1, 48, num_layers=1, batch_first=True)
            self.static = nn.Sequential(nn.Linear(static_size, 16), nn.ReLU(), nn.LayerNorm(16))
            self.head   = nn.Linear(48 + 16, 1)
            self.drop   = nn.Dropout(0.3)
        def forward(self, xs, xc):
            _, (h, _) = self.lstm(xs)
            return self.head(torch.cat([self.drop(h.squeeze(0)), self.static(xc)], dim=1))

    tr_dl = DataLoader(DualDS(y_tr, cal_tr, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=True)
    va_dl = DataLoader(DualDS(y_va, cal_va, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=False)

    model   = LSTMModel().to(DEVICE)
    opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_val, no_imp, best_state = np.inf, 0, None

    for ep in range(EPOCHS):
        model.train()
        for xs, xc, yt in tr_dl:
            opt.zero_grad()
            loss_fn(model(xs.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).backward()
            opt.step()
        model.eval()
        vl = sum(loss_fn(model(xs.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).item()
                 for xs, xc, yt in va_dl)
        if vl < best_val:
            best_val, no_imp = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    y_ctx   = np.concatenate([y_va, y_te])
    cal_ctx = np.vstack([cal_va, cal_te])
    buf, preds = list(y_tr[-SEQ_LEN:]), []
    with torch.no_grad():
        for i in range(len(y_ctx)):
            xs = torch.tensor(np.array(buf[-SEQ_LEN:]).reshape(1, SEQ_LEN, 1),
                               dtype=torch.float32).to(DEVICE)
            xc = torch.tensor(cal_ctx[i].reshape(1, -1),
                               dtype=torch.float32).to(DEVICE)
            preds.append(model(xs, xc).item())
            buf.append(y_ctx[i])

    return pd.Series(
        y_sc.inverse_transform(np.array(preds[len(y_va):]).reshape(-1, 1)).ravel().clip(0),
        index=test.index)


## 8. N-BEATS v2 (no history features)
Covariate-conditioned N-BEATS:
- **Lookback window:** 30 days of raw `avg_kwh`
- **Covariate vector:** 9 calendar features only


In [ ]:
def run_nbeats_v2(train, val, test, df_feats):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

    y_sc  = StandardScaler()
    y_tr  = y_sc.fit_transform(train[[TARGET_COL]]).ravel()
    y_va  = y_sc.transform(val[[TARGET_COL]]).ravel()
    y_te  = y_sc.transform(test[[TARGET_COL]]).ravel()

    cal_tr = _cal_array(train, df_feats)
    cal_va = _cal_array(val,   df_feats)
    cal_te = _cal_array(test,  df_feats)
    cal_size = cal_tr.shape[1]

    class WinDS(Dataset):
        def __init__(self, y, cal, seq_len):
            self.y, self.cal, self.seq_len = y, cal, seq_len
        def __len__(self):
            return max(0, len(self.y) - self.seq_len)
        def __getitem__(self, i):
            return (torch.tensor(self.y[i:i+self.seq_len], dtype=torch.float32),
                    torch.tensor(self.cal[i+self.seq_len], dtype=torch.float32),
                    torch.tensor([self.y[i+self.seq_len]], dtype=torch.float32))

    class NBEATSBlock(nn.Module):
        def __init__(self, in_sz, cov_sz, hidden):
            super().__init__()
            self.fc = nn.Sequential(
                nn.Linear(in_sz + cov_sz, hidden), nn.ReLU(), nn.Dropout(0.1),
                nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.1))
            self.back = nn.Linear(hidden, in_sz)
            self.fore = nn.Linear(hidden, 1)
        def forward(self, x, cov):
            h = self.fc(torch.cat([x, cov], dim=-1))
            return self.back(h), self.fore(h)

    class NBEATSModel(nn.Module):
        def __init__(self, n_blocks=3, hidden=64):
            super().__init__()
            self.blocks = nn.ModuleList(
                [NBEATSBlock(SEQ_LEN, cal_size, hidden) for _ in range(n_blocks)])
        def forward(self, x, cov):
            res, forecast = x, torch.zeros(x.size(0), 1, device=x.device)
            for blk in self.blocks:
                back, f = blk(res, cov)
                res      = res - back
                forecast = forecast + f
            return forecast

    tr_dl = DataLoader(WinDS(y_tr, cal_tr, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=True)
    va_dl = DataLoader(WinDS(y_va, cal_va, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=False)

    model   = NBEATSModel().to(DEVICE)
    opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_val, no_imp, best_state = np.inf, 0, None

    for ep in range(EPOCHS):
        model.train()
        for xb, xc, yt in tr_dl:
            opt.zero_grad()
            loss_fn(model(xb.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).backward()
            opt.step()
        model.eval()
        vl = sum(loss_fn(model(xb.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).item()
                 for xb, xc, yt in va_dl)
        if vl < best_val:
            best_val, no_imp = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    y_ctx   = np.concatenate([y_va, y_te])
    cal_ctx = np.vstack([cal_va, cal_te])
    buf, preds = list(y_tr[-SEQ_LEN:]), []
    with torch.no_grad():
        for i in range(len(y_ctx)):
            xb = torch.tensor(np.array(buf[-SEQ_LEN:]).reshape(1, -1),
                               dtype=torch.float32).to(DEVICE)
            xc = torch.tensor(cal_ctx[i].reshape(1, -1),
                               dtype=torch.float32).to(DEVICE)
            preds.append(model(xb, xc).item())
            buf.append(y_ctx[i])

    return pd.Series(
        y_sc.inverse_transform(np.array(preds[len(y_va):]).reshape(-1, 1)).ravel().clip(0),
        index=test.index)


## 9. Hybrid v2 (no history features)
Seasonal Naive (365d) + ResidualLSTM.
The residual LSTM uses only calendar features as the static context — no engineered history.


In [ ]:
def run_hybrid_v2(train, val, test, df_feats):
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

    # Step 1: Seasonal Naive baseline prediction on test set
    sn_pred = run_seasonal_naive(pd.concat([train, val]), test)

    # Step 2: Compute seasonal naive residuals on train+val inline
    tv      = pd.concat([train, val])
    tv_vals = tv[TARGET_COL].values
    sn_tv   = [float(tv_vals[i - SEASONAL_PERIOD]) if i >= SEASONAL_PERIOD
               else float(tv_vals[0])
               for i in range(len(tv_vals))]
    resid_tv = tv_vals - np.array(sn_tv)

    # Scale residuals
    sc   = StandardScaler()
    r_tr = sc.fit_transform(resid_tv[:len(train)].reshape(-1, 1)).ravel()
    r_va = sc.transform(resid_tv[len(train):].reshape(-1, 1)).ravel()

    cal_tr   = _cal_array(train, df_feats)
    cal_va   = _cal_array(val,   df_feats)
    cal_te   = _cal_array(test,  df_feats)
    cal_size = cal_tr.shape[1]

    class RDS(Dataset):
        def __init__(self, r, cal, seq_len):
            self.r, self.cal, self.seq_len = r, cal, seq_len
        def __len__(self):
            return max(0, len(self.r) - self.seq_len)
        def __getitem__(self, i):
            return (torch.tensor(self.r[i:i+self.seq_len].reshape(-1, 1), dtype=torch.float32),
                    torch.tensor(self.cal[i+self.seq_len], dtype=torch.float32),
                    torch.tensor([self.r[i+self.seq_len]], dtype=torch.float32))

    class ResLSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm   = nn.LSTM(1, 32, batch_first=True)
            self.static = nn.Sequential(nn.Linear(cal_size, 16), nn.ReLU())
            self.head   = nn.Linear(32 + 16, 1)
            self.drop   = nn.Dropout(0.3)
        def forward(self, xs, xc):
            _, (h, _) = self.lstm(xs)
            return self.head(torch.cat([self.drop(h.squeeze(0)), self.static(xc)], dim=1))

    tr_dl = DataLoader(RDS(r_tr, cal_tr, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=True)
    va_dl = DataLoader(RDS(r_va, cal_va, SEQ_LEN), batch_size=BATCH_SIZE, shuffle=False)

    model   = ResLSTM().to(DEVICE)
    opt     = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_val, no_imp, best_state = np.inf, 0, None

    for ep in range(EPOCHS):
        model.train()
        for xs, xc, yt in tr_dl:
            opt.zero_grad()
            loss_fn(model(xs.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).backward()
            opt.step()
        model.eval()
        vl = sum(loss_fn(model(xs.to(DEVICE), xc.to(DEVICE)), yt.to(DEVICE)).item()
                 for xs, xc, yt in va_dl)
        if vl < best_val:
            best_val, no_imp = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()

    r_ctx   = np.concatenate([r_va, np.zeros(len(test))])
    cal_ctx = np.vstack([cal_va, cal_te])
    buf, preds_r = list(r_tr[-SEQ_LEN:]), []
    with torch.no_grad():
        for i in range(len(r_ctx)):
            xs = torch.tensor(np.array(buf[-SEQ_LEN:]).reshape(1, SEQ_LEN, 1),
                               dtype=torch.float32).to(DEVICE)
            xc = torch.tensor(cal_ctx[i].reshape(1, -1),
                               dtype=torch.float32).to(DEVICE)
            p = model(xs, xc).item()
            buf.append(r_ctx[i] if i < len(r_va) else p)
            preds_r.append(p)

    resid_test = sc.inverse_transform(np.array(preds_r[len(r_va):]).reshape(-1, 1)).ravel()
    return pd.Series((sn_pred.values + resid_test).clip(0), index=test.index)


## 10. Run Pipeline for All Cities

Runs all 9 models per city, saves per-city `*_metrics.csv` and `*_predictions.csv`.


In [ ]:
def run_city(city_name, df_city):
    print(f"\n{'='*60}\n  {city_name}\n{'='*60}")
    df_city = df_city.sort_values(DATE_COL).set_index(DATE_COL)
    df_city = add_calendar_features(df_city.reset_index()).set_index(DATE_COL)
    train, val, test = split_series(df_city)
    tv = pd.concat([train, val])
    print(f"  Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

    results, pred_dict = {}, {"date": test.index, "actual": test[TARGET_COL].values}

    def _run(name, fn, col):
        t0 = time.time()
        try:
            pred = fn().reindex(test.index)
            mae, rmse, mp, smp = metrics(test[TARGET_COL].values, pred.values)
            results[name]  = {"MAE": mae, "RMSE": rmse, "MAPE(%)": mp, "sMAPE(%)": smp}
            pred_dict[col] = pred.values
            elapsed = time.time() - t0
            print(f"  {name:<45} MAPE={mp:.2f}%  ({elapsed:.1f}s)")
        except Exception as e:
            print(f"  {name:<45} FAILED: {e}")
            results[name]  = dict(MAE=float("nan"), RMSE=float("nan"),
                                  **{"MAPE(%)": float("nan"), "sMAPE(%)": float("nan")})
            pred_dict[col] = np.full(len(test), float("nan"))

    _run("Naive",                          lambda: run_naive(tv, test),                "Naive")
    _run("Seasonal Naive (365d)",          lambda: run_seasonal_naive(tv, test),       "Seasonal Naive (365d)")
    _run("Rolling Mean (7d)",              lambda: run_rolling_mean(tv, test),         "Rolling Mean (7d)")
    _run("SARIMA",                         lambda: run_sarima(train, val, test),       "SARIMA")
    _run("Ridge (calendar only)",          lambda: run_ridge(train, val, test, df_city),    "Ridge (calendar only)")
    _run("Random Forest (calendar only)",  lambda: run_rf(train, val, test, df_city),       "Random Forest (calendar only)")
    _run("LSTM v2 (no history)",           lambda: run_lstm_v2(train, val, test, df_city),  "LSTM v2 (no history)")
    _run("N-BEATS v2 (no history)",        lambda: run_nbeats_v2(train, val, test, df_city),"N-BEATS v2 (no history)")
    _run("Hybrid v2 (no history)",         lambda: run_hybrid_v2(train, val, test, df_city),"Hybrid v2 (no history)")

    slug = city_name.replace("/", "_").replace(" ", "_")
    pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"}).to_csv(
        OUT_DIR / f"{slug}_metrics.csv", index=False)
    pd.DataFrame(pred_dict).to_csv(OUT_DIR / f"{slug}_predictions.csv", index=False)
    return results

print("Loading data...")
df_all = pd.read_csv(DATA_FILE, parse_dates=[DATE_COL])
df_all = df_all[df_all[GROUP_COL].isin(MUNICIPALITIES)]

all_results = {}
for city in MUNICIPALITIES:
    all_results[city] = run_city(city, df_all[df_all[GROUP_COL] == city].copy())


## 11. Summary & Comparison

MAPE summary across all cities, and side-by-side comparison with the full-feature run (notebook 4).


In [ ]:
# Summary table
rows = []
for city, res in all_results.items():
    for model, m in res.items():
        rows.append({"Municipality": city, "Model": model, **m})
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_no_features.csv", index=False)

pivot = summary.pivot_table(values="MAPE(%)", index="Model", columns="Municipality")
print("\nMAPE (%) — No Feature Engineering (calendar only)")
print("="*70)
print(pivot.round(2).to_string())
print(f"\nResults saved to: {OUT_DIR}")


In [ ]:
# Side-by-side comparison vs full-feature run (notebook 4)
fe_base = BASE_PATH / "forecasting_model_results_365_seasonality_v4_avg_kwh"

name_map = {
    "Naive":                          "Naive",
    "Seasonal Naive (365d)":          "Seasonal Naive (365d)",
    "Rolling Mean (7d)":              "Rolling Mean (7d)",
    "SARIMA":                         "SARIMAX",
    "Ridge (calendar only)":          "Ridge Features",
    "Random Forest (calendar only)":  "Random Forest Features",
    "LSTM v2 (no history)":           "LSTM v2 (dual-stream)",
    "N-BEATS v2 (no history)":        "N-BEATS v2 (covariate-conditioned)",
    "Hybrid v2 (no history)":         "Hybrid v2 SeasonalNaive(365d)+ResidualLSTM",
}

city_slugs = {
    "Vitoria-Gasteiz":        "Vitoria-Gasteiz",
    "Donostia/San Sebastian": "Donostia_San_Sebastian",
    "Pamplona/Iruna":         "Pamplona_Iruna",
}

comp_rows = []
for nf_name, fe_name in name_map.items():
    row = {"Model": nf_name}
    for city_label, slug in city_slugs.items():
        nf_df = pd.read_csv(OUT_DIR  / f"{slug}_metrics.csv").set_index("Model")
        fe_df = pd.read_csv(fe_base  / f"{slug}_metrics.csv").set_index("Model")
        nf_v  = nf_df.loc[nf_name, "MAPE(%)"] if nf_name in nf_df.index else float("nan")
        fe_v  = fe_df.loc[fe_name,  "MAPE(%)"] if fe_name  in fe_df.index else float("nan")
        short = city_label.split("/")[0][:8]
        row[f"{short} No FE"]  = round(nf_v, 2)
        row[f"{short} With FE"] = round(fe_v, 2)
        row[f"{short} Delta"]   = round(fe_v - nf_v, 2)
    comp_rows.append(row)

cmp = pd.DataFrame(comp_rows).set_index("Model")
print("\nDelta = With FE − No FE  (negative = feature engineering helps)")
print(cmp.to_string())
